# LangGraph 012 — A Streamlit UI for the Chatbot

Streamlit apps are usually judged by eye. They can be measured instead:
`streamlit.testing.v1.AppTest` runs a real app script, types into the real
`st.chat_input`, and reports what the app rendered — headlessly, with no
browser and no API key.

| Part | What we check |
|---|---|
| A | the rerun: a plain variable against `session_state` |
| B | 2 of 6 messages against 6 of 6 |
| C | the LangGraph backend behind the UI |
| D | the missing-`config` error |
| E | two memories, drifting in both directions |
| F | one hard-coded `thread_id`, two users |

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from streamlit.testing.v1 import AppTest

TURNS = ["Hi, my name is Ayesha", "What is my name?", "Where is Kerala?"]

def say(app, turns=TURNS):
    for turn in turns:
        app.chat_input[0].set_value(turn).run()
    return app

def rendered(app):
    return [(m.proto.name, m.children[0].value) for m in app.chat_message]

print("AppTest ready - every figure below is a real app run")

## Part A — The rerun

In [ ]:
PLAIN = """
import streamlit as st

turns = 0                       # re-created on every rerun
user = st.chat_input("Type here")
if user:
    turns += 1
st.text(f"turns: {turns}")
"""

SESSION = """
import streamlit as st

if "turns" not in st.session_state:
    st.session_state["turns"] = 0          # runs once per session
user = st.chat_input("Type here")
if user:
    st.session_state["turns"] += 1
st.text(f"turns: {st.session_state['turns']}")
"""

for label, source in (("plain variable", PLAIN), ("session_state", SESSION)):
    app = say(AppTest.from_string(source).run())
    print(f"{label:<16} after 3 messages -> {app.text[0].value!r}")

assert say(AppTest.from_string(PLAIN).run()).text[0].value == "turns: 1"
assert say(AppTest.from_string(SESSION).run()).text[0].value == "turns: 3"
print()
print("The script ran four times in both cases - once at load, once per")
print("message. Only session_state carried anything between them.")

## Part B — The chat history

In [ ]:
BROKEN = """
import streamlit as st

message_history = []            # WRONG place: reset on every rerun

for message in message_history:
    with st.chat_message(message["role"]):
        st.text(message["content"])

user_input = st.chat_input("Type here")
if user_input:
    message_history.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.text(user_input)
    message_history.append({"role": "assistant", "content": user_input})
    with st.chat_message("assistant"):
        st.text(user_input)
"""

FIXED = """
import streamlit as st

if "message_history" not in st.session_state:
    st.session_state["message_history"] = []

for message in st.session_state["message_history"]:
    with st.chat_message(message["role"]):
        st.text(message["content"])

user_input = st.chat_input("Type here")
if user_input:
    st.session_state["message_history"].append(
        {"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.text(user_input)
    st.session_state["message_history"].append(
        {"role": "assistant", "content": user_input})
    with st.chat_message("assistant"):
        st.text(user_input)
"""

for label, source in (("plain list", BROKEN), ("session_state", FIXED)):
    app = say(AppTest.from_string(source).run())
    shown = rendered(app)
    print(f"{label:<14} 3 messages typed -> {len(shown)} rendered")
    for role, text in shown:
        print(f"    {role:<10} {text!r}")

assert len(rendered(say(AppTest.from_string(BROKEN).run()))) == 2
assert len(rendered(say(AppTest.from_string(FIXED).run()))) == 6
print()
print("2 of 6 against 6 of 6. The append was always correct; the line that")
print("broke it was the assignment above it.")

## Part C — The backend behind the UI

Two real files, written to a temporary directory: the lesson 010 graph with a
scripted model, and the frontend that imports it.

In [ ]:
import tempfile, textwrap
from pathlib import Path

BACKEND = textwrap.dedent("""
    import re
    from typing import Annotated, TypedDict
    from langchain_core.messages import AIMessage, BaseMessage
    from langgraph.checkpoint.memory import InMemorySaver
    from langgraph.graph import END, START, StateGraph
    from langgraph.graph.message import add_messages

    FACTS = {"kerala": "Kerala is in the south-west of India."}

    class ChatState(TypedDict):
        messages: Annotated[list[BaseMessage], add_messages]

    def scripted_llm(messages):
        text = messages[-1].content.lower()
        history = " ".join(m.content for m in messages[:-1])
        if "my name is" in text:
            name = messages[-1].content.split("my name is")[-1].strip(" .!?")
            return AIMessage(content=f"Hello {name}, nice to meet you.")
        if "my name" in text:
            found = re.search(r"my name is ([A-Z][a-z]+)", history)
            return AIMessage(content=f"Your name is {found.group(1)}." if found
                             else "I do not have access to that.")
        for key, answer in FACTS.items():
            if key in text:
                return AIMessage(content=answer)
        return AIMessage(content="Noted.")

    def chat_node(state: ChatState) -> dict:
        return {"messages": [scripted_llm(state["messages"])]}

    _g = StateGraph(ChatState)
    _g.add_node("chat_node", chat_node)
    _g.add_edge(START, "chat_node")
    _g.add_edge("chat_node", END)
    chatbot = _g.compile(checkpointer=InMemorySaver())
""")

BACKEND_DIR = Path(tempfile.mkdtemp(prefix="lg012_"))
(BACKEND_DIR / "langgraph_backend.py").write_text(BACKEND, encoding="utf-8")

FRONTEND = """
import sys
sys.path.insert(0, r"{backend_dir}")

import streamlit as st
from langchain_core.messages import HumanMessage
from langgraph_backend import chatbot

CONFIG = {{"configurable": {{"thread_id": "{thread_id}"}}}}

if "message_history" not in st.session_state:
    st.session_state["message_history"] = []

for message in st.session_state["message_history"]:
    with st.chat_message(message["role"]):
        st.text(message["content"])

user_input = st.chat_input("Type here")
if user_input:
    st.session_state["message_history"].append(
        {{"role": "user", "content": user_input}})
    with st.chat_message("user"):
        st.text(user_input)

    response = chatbot.invoke({{"messages": [HumanMessage(content=user_input)]}},
                              {config_arg})
    ai_message = response["messages"][-1].content

    st.session_state["message_history"].append(
        {{"role": "assistant", "content": ai_message}})
    with st.chat_message("assistant"):
        st.text(ai_message)
"""

def frontend(thread_id="thread-1", with_config=True):
    return FRONTEND.format(backend_dir=BACKEND_DIR, thread_id=thread_id,
                           config_arg="config=CONFIG" if with_config else "")

app = say(AppTest.from_string(frontend()).run())
for role, text in rendered(app):
    print(f"  {role:<10} {text!r}")
answers = [t for r, t in rendered(app) if r == "assistant"]
assert "Ayesha" in answers[1]
print()
print("The model knew the name because the checkpointer handed it the earlier")
print("messages - not because the page displayed them.")

## Part D — The config is not optional

In [ ]:
import contextlib, io

buffer = io.StringIO()
with contextlib.redirect_stdout(buffer), contextlib.redirect_stderr(buffer):
    broken = AppTest.from_string(frontend(with_config=False)).run()
    broken.chat_input[0].set_value("Hi").run()

assert broken.exception
first = broken.exception[0]
print(getattr(first, "type", "Exception"), ":",
      (getattr(first, "value", "") or "").splitlines()[0][:90])
print()
print("A checkpointer-compiled graph has nowhere to file the state without a")
print("thread, so the very first message fails.")

## Part E — Two memories

`session_state["message_history"]` is what the screen shows. The checkpointer's
thread is what the model sees. Nothing keeps them in step.

In [ ]:
source = frontend(thread_id="pair-A")
app = say(AppTest.from_string(source).run())
print("three turns ->", len(rendered(app)), "on screen,",
      len(app.session_state["message_history"]), "in session_state")

# the page is refreshed: session_state goes, the thread_id stays
refreshed = AppTest.from_string(source).run()
print("refreshed   ->", len(rendered(refreshed)), "on screen")
refreshed.chat_input[0].set_value("What is my name?").run()
answer = rendered(refreshed)[-1][1]
print("ask again   ->", repr(answer))
assert len(rendered(AppTest.from_string(source).run())) == 0
assert "Ayesha" in answer

# the mirror image: keep the screen, change the thread
other = AppTest.from_string(frontend(thread_id="pair-B")).run()
other.session_state["message_history"] = [
    {"role": "user", "content": "Hi, my name is Ayesha"},
    {"role": "assistant", "content": "Hello Ayesha, nice to meet you."}]
other.chat_input[0].set_value("What is my name?").run()
mirrored = rendered(other)[-1][1]
print("new thread, old screen ->", repr(mirrored))
assert "Ayesha" not in mirrored
print()
print("A blank page with a model that remembers, and a displayed conversation")
print("the model never had. Neither raises an error.")

## Part F — One hard-coded thread, two users

In [ ]:
shared = frontend(thread_id="thread-1")

user_a = AppTest.from_string(shared).run()
user_a.chat_input[0].set_value("Hi, my name is Ayesha").run()

user_b = AppTest.from_string(shared).run()          # a different visitor
user_b.chat_input[0].set_value("What is my name?").run()
leaked = rendered(user_b)[-1][1]
print("user B, on a blank page, is told:", repr(leaked))
print("user B's screen holds", len(rendered(user_b)), "messages")
assert "Ayesha" in leaked
print()
print("The backend module - and its checkpointer - is imported once per server,")
print("so a constant thread_id is one shared conversation. The leak is in the")
print("model's answer, not in the UI.")

In [ ]:
# the fix: a thread id per session, created with the history it belongs with
PER_SESSION = frontend(thread_id="PLACEHOLDER").replace(
    'CONFIG = {"configurable": {"thread_id": "PLACEHOLDER"}}',
    'import uuid\n'
    'if "thread_id" not in st.session_state:\n'
    '    st.session_state["thread_id"] = str(uuid.uuid4())\n'
    'CONFIG = {"configurable": {"thread_id": st.session_state["thread_id"]}}')

a = AppTest.from_string(PER_SESSION).run()
a.chat_input[0].set_value("Hi, my name is Ayesha").run()
b = AppTest.from_string(PER_SESSION).run()
b.chat_input[0].set_value("What is my name?").run()
isolated = rendered(b)[-1][1]
print("user B now hears:", repr(isolated))
print("their thread ids differ:", a.session_state["thread_id"] != b.session_state["thread_id"])
assert "Ayesha" not in isolated
assert a.session_state["thread_id"] != b.session_state["thread_id"]
print()
print("Four lines, and both problems are gone: the thread and the display")
print("history are now created and cleared together.")

## What this notebook established

- Streamlit re-runs the whole script on every interaction: a counter in a plain
  variable read **1** after three messages, in `session_state` **3**.
- The same chat app rendered **2 of 6** messages with its history in a plain
  list and **6 of 6** with it in `session_state`.
- Connecting the LangGraph backend changes one block, and the model answered
  *What is my name?* through the checkpointer rather than the page.
- Omitting `config` fails on the first message: *Checkpointer requires one or
  more of the following 'configurable' keys*.
- The screen's history and the model's history drift in **both** directions —
  blank page with a remembering model, and a displayed conversation the model
  never had.
- A hard-coded `thread_id` let user B be told user A's name. A `uuid4` in
  `session_state` fixed both problems in four lines.

## Exercises

1. Add a sidebar button that starts a new conversation. What has to be cleared,
   and in what order?
2. Stream the reply with `st.write_stream` and the graph's `stream()`. Does the
   history still end up correct?
3. Build the display history **from** the checkpointer on first load, so the two
   can never disagree. What breaks if the thread is empty?
4. Write an AppTest that fails if the thread id is ever a constant.